# CancerEmo Dataset Exploration

## 1 Overview

Before pre-trained text-emotion models are evaluated, this notebook examines the CancerEmo dataset. The dataset, which is separated into eight distinct emotion files, includes sentences gathered from cancer-related online discussions [1].

To indicate if an emotion is present in a statement, each file uses a binary label. The file structure, dataset splits, emotion distribution, missing values, duplicate sentences, and empty text are all evaluated. Since the dataset will only be used for model evaluation, the original files are left unmodified.

## 2 Importing Libraries

To find the CSV files and arrange their text and label information, the necessary libraries are imported.

In [1]:
# Importing required libraries
from pathlib import Path
import pandas as pd

## 3 Loading Dataset

Since each of the eight CancerEmo CSV files represents a single emotion, they are imported independently. Keeping them separate at this stage allows their columns, record counts and binary labels to be checked before the data is prepared for model evaluation.

In [2]:
# folder
path = Path("../../data/raw/canceremo")

In [3]:
# emotion files
files = {
    "Anger": "Anger_anon.csv",
    "Anticipation": "Anticipation_anon.csv",
    "Disgust": "Disgust_anon.csv",
    "Fear": "Fear_anon.csv",
    "Joy": "Joy_anon.csv",
    "Sadness": "Sadness_anon.csv",
    "Surprise": "Surprise_anon.csv",
    "Trust": "Trust_anon.csv"
}

In [4]:
# dictionary
emotion_datasets = {}

In [5]:
# for each file total records
for emotion, filename in files.items():
    # file path and read the csv
    file_path = path / filename
    emotion_datasets[emotion] = pd.read_csv(file_path)
    print(f"{emotion}: {len(emotion_datasets[emotion])} records")

Anger: 837 records
Anticipation: 436 records
Disgust: 891 records
Fear: 5388 records
Joy: 6043 records
Sadness: 3606 records
Surprise: 826 records
Trust: 1887 records


To verify that the split columns, emotion label, and phrase were loaded correctly, the first 5 data from each file are shown.

In [6]:
# first 5 records
for emotion, dataset in emotion_datasets.items():
    display(dataset.head(5))

,Sentence,Anger,Split
0,And it will no doubt make me happy in the morn...,0,0
1,My dr gave me a rx for morphine oral to take f...,0,0
2,surgeon did broncos copy on upper lobe of left...,0,0
3,The apex seems free of trouble.,0,0
4,Good morning..I have been reading this post an...,0,0


,Sentence,Anticipation,Split
0,Isn't it wonderful to have something to rejoic...,0,0
1,I dont have lymphadema but I do get PT.,0,0
2,I am going to have the radiation was just conf...,0,0
3,I have met with my pulmonologist several times...,0,0
4,The procedure sounds very promising with suppo...,0,0


,Sentence,Disgust,Split
0,I would like to know if anyone else has had an...,0,0
1,"Hey <PERSON>, When I was on Taxotere everythin...",0,0
2,He has planned on 6 rounds prior to maintenanc...,0,0
3,Unfortunately the chemo hasn't worked and the ...,0,0
4,Enjoy whatever foods you want now!,0,0


,Sentence,Fear,Split
0,Hopefully we have more good days than we have ...,0,0
1,I explained that my own self esteem would be c...,0,0
2,We've been told that some physicians may take ...,0,0
3,They keep coming in every hour or so to do neu...,0,0
4,It has proven effective and improves the quali...,0,0


,Sentence,Joy,Split
0,The university teaching hospitals with cancer ...,0,0
1,We've been told that some physicians may take ...,0,0
2,With reference to MUSE - my question was : Doe...,0,0
3,"However, new symptoms have doctors insisting i...",0,0
4,"I wonder, where do you live?",0,0


,Sentence,Sadness,Split
0,May all your wishes come true and may you have...,0,0
1,With reference to MUSE - my question was : Doe...,0,0
2,They keep coming in every hour or so to do neu...,0,0
3,My wife is the one with stage 4 lung cancer an...,0,0
4,"However, it will be a few days before we get t...",0,0


,Sentence,Surprise,Split
0,I even watched how to make a scarf out of a te...,0,0
1,I believe the scoring is from 0 to 6.,0,0
2,I was advised to take 1 Claritin and 2 Tyleno...,0,0
3,Hope your day is filled with joy and happiness...,0,0
4,"I wonder, where do you live?",0,0


,Sentence,Trust,Split
0,My body is sick of being sick and tired.,0,0
1,I think I like the 4 hour one the best!,0,0
2,All I knew about cancer when I was first diagn...,0,0
3,"BETWEEN THE TWO CANCERS, I AM SO CONFUSED AND ...",0,0
4,The radiologist said due to the pattern it did...,0,0


Each of the eight CancerEmo files was loaded independently. A text, a binary label for its emotion, and a predetermined split value are all included in each file. Before any evaluation data is chosen, the files will undergo additional inspection.

## 4 Understanding Dataset Split

CancerEmo already separates its data into sets for testing, validation, and training. 0 is used for training, 1 for validation, and 2 for testing in the Split column. To prevent evaluation data from being combined with other records, these original splits are examined and kept intact.

In [7]:
# list to store the split results
split_results = []

In [8]:
for emotion, dataset in emotion_datasets.items():
    # split count for each csv
    split_counts = dataset["Split"].value_counts()
    # append
    split_results.append({
        "Emotion": emotion,
        "Training": split_counts.get(0, 0),
        "Validation": split_counts.get(1, 0),
        "Testing": split_counts.get(2, 0)
    })

In [9]:
# check the distribution of the data
df = pd.DataFrame(split_results)
df

,Emotion,Training,Validation,Testing
0,Anger,669,84,84
1,Anticipation,360,34,42
2,Disgust,735,90,66
3,Fear,4310,539,539
4,Joy,4834,604,605
5,Sadness,2884,361,361
6,Surprise,614,102,110
7,Trust,1509,189,189


For every emotion file, the training, validation, and testing records were located. The first splits will not change. While the testing data will only be used for the final evaluation of the chosen model, the validation records will be used to compare the candidate pre-trained text models. CancerEmo will not be used for model training since we are using pre-trained models for this project.

## 5 Class Distribution

When an emotion is present, each CancerEmo file uses 1 and when it is not, it uses 0. Because unequal labels can impact model evaluation, the amount of positive and negative records is verified in the testing and validation splits.

In [10]:
# label distribution
label_distribution = []

In [11]:
for emotion, dataset in emotion_datasets.items():
    # label distribution check
    validation_data = dataset[dataset["Split"] == 1]
    testing_data = dataset[dataset["Split"] == 2]
    # append
    label_distribution.append({
        "Emotion": emotion,
        "Validation Negative": (validation_data[emotion] == 0).sum(),
        "Validation Positive": (validation_data[emotion] == 1).sum(),
        "Testing Negative": (testing_data[emotion] == 0).sum(),
        "Testing Positive": (testing_data[emotion] == 1).sum()
    })

In [12]:
# label dataframe
df_label = pd.DataFrame(label_distribution)
df_label

,Emotion,Validation Negative,Validation Positive,Testing Negative,Testing Positive
0,Anger,43,41,50,34
1,Anticipation,17,17,21,21
2,Disgust,45,45,33,33
3,Fear,279,260,253,286
4,Joy,293,311,304,301
5,Sadness,165,196,181,180
6,Surprise,51,51,55,55
7,Trust,94,95,96,93


Across the testing and validation splits, the positive and negative labels are fairly balanced. As a result, neither sampling nor balancing are necessary. To ensure a fair model comparison, accuracy will still be combined with Macro-F1, precision, and recall.

## 6 Checking Text and Label Data

Every emotion file is checked for duplicate sentences, invalid binary labels, missing values, and empty phrases. These tests help ensure that the text-model evaluation is not impacted by incomplete or repetitive records.

In [13]:
# list for the checkings
checks = []

In [14]:
for emotion, dataset in emotion_datasets.items():
    # missing sentences
    missing_sentences = dataset["Sentence"].isnull().sum()
    # empty sentences
    empty_sentences = dataset["Sentence"].fillna("").str.strip().eq("").sum()
    # invalid values
    invalid_labels = (~dataset[emotion].isin([0, 1])).sum()
    invalid_splits = (~dataset["Split"].isin([0, 1, 2])).sum()
    # duplicates
    duplicated_sentences = dataset["Sentence"].duplicated().sum()
    # append 
    checks.append({
        "Emotion": emotion,
        "Missing Sentences": missing_sentences,
        "Empty Sentences": empty_sentences,
        "Invalid Labels": invalid_labels,
        "Invalid Splits": invalid_splits,
        "Duplicated Sentences": duplicated_sentences
    })

In [15]:
# dataframe for the checks
df_checks = pd.DataFrame(checks)
df_checks

,Emotion,Missing Sentences,Empty Sentences,Invalid Labels,Invalid Splits,Duplicated Sentences
0,Anger,0,0,0,0,0
1,Anticipation,0,0,0,0,0
2,Disgust,0,0,0,0,0
3,Fear,0,0,0,0,0
4,Joy,0,0,0,0,0
5,Sadness,0,0,0,0,0
6,Surprise,0,0,0,0,0
7,Trust,0,0,0,0,0


None of the eight emotion files contained any missing sentences, empty sentences, invalid labels, invalid split values, or duplicate phrases. Therefore, before model evaluation, no records need to be deleted or modified.

Anger, anticipation, disgust, fear, joy, sadness, surprise, and trust are the eight distinct binary categorisation tasks that make up CancerEmo. A value of 1 indicates the presence of each emotion, while a value of 0 indicates its absence. The testing split will be applied for the final evaluation of the chosen model, while the validation split will be used to compare potential text models. The primary comparison outcome will be the average macro-F1 over the eight tasks.

## 7 Evaluation Splits

For the purpose of evaluating the model later, the testing and validation records remain separate. The testing data will be saved for the final evaluation of the chosen model, and the validation data will be used to compare the candidate pre-trained text models. Since no model training or fine-tuning is done, the training records are not used.

In [16]:
# dictionaries for validation and testing datasets
validation_datasets = {}
testing_datasets = {}

In [17]:
# count the evaluation split
evaluationsplit = []

In [18]:
for emotion, dataset in emotion_datasets.items():
    # validation dataset
    validation_datasets[emotion] = (dataset.loc[dataset["Split"] == 1, ["Sentence", emotion]].reset_index(drop=True))
    # testing dataset
    testing_datasets[emotion] = (dataset.loc[dataset["Split"] == 2, ["Sentence", emotion]].reset_index(drop=True))
    # append
    evaluationsplit.append({
        "Emotion": emotion,
        "Validation Records": len(validation_datasets[emotion]),
        "Testing Records": len(testing_datasets[emotion])
    })

In [19]:
df_evaluation = pd.DataFrame(evaluationsplit)
df_evaluation

,Emotion,Validation Records,Testing Records
0,Anger,84,84
1,Anticipation,34,42
2,Disgust,90,66
3,Fear,539,539
4,Joy,604,605
5,Sadness,361,361
6,Surprise,102,110
7,Trust,189,189


For every emotion, the validation and testing records were effectively separated. These datasets stay the same, and as CancerEmo is used only to assess pre-trained models, the training records are not included. 

The prepared testing and validation records are stored in several CSV files. By doing this, the dataset preparation procedure is not repeated when the candidate text-emotion models are subsequently evaluated. The original CancerEmo files are still intact.

In [20]:
# path for the processed folder
processed_path = Path("../../data/processed/canceremo")

In [21]:
# validation and testing folder
validation_folder = processed_path / "validation"
testing_folder = processed_path / "testing"

In [22]:
# these folders are created if not create
validation_folder.mkdir(parents=True, exist_ok=True)
testing_folder.mkdir(parents=True, exist_ok=True)

In [23]:
for emotion in emotion_datasets:
    # filename
    filename = f"{emotion.lower()}.csv"
    # validaiton and testing csv in their folders
    validation_datasets[emotion].to_csv(validation_folder / filename, index=False)
    testing_datasets[emotion].to_csv(testing_folder / filename, index=False)

In [24]:
# print statements
print("Validation files:", len(validation_datasets))
print("Testing files:", len(testing_datasets))

Validation files: 8
Testing files: 8


The CancerEmo data that has been processed was arranged into several folders for testing and validation. There is a CSV file for each of the eight emotions in each folder. This structure preserves the original dataset while separating the model-selection data from the final assessment data.

## 8 Conclusion

All eight emotion categories anger, anticipation, disgust, fear, joy, sadness, surprise, and trust were effectively examined in the CancerEmo dataset. There were no duplicate sentences, invalid split values, invalid labels, missing sentences, or empty text in the dataset. Additionally, the validation and testing splits had a fair distribution of positive and negative labels.

There was no need for text modification, cleaning, or balance. The original dataset divides were maintained, with the testing data kept aside for the final evaluation of the chosen model and the validation data prepared for comparing potential pre-trained text-emotion models. The original CancerEmo files were kept unchanged, however the processed files were saved in different validation and testing folders.

## References
[1] T. Sosea and C. Caragea, "CancerEmo: A dataset for fine-grained emotion detection," 2020. [Online].
Available: https://aclanthology.org/2020.emnlp-main.715/. [Accessed: Jun 21, 2026].